## ML_1035: Flash Attention (Tiled)

**Difficulty**: Hard | **TorchCode**: #25

### Problem
Implement tiled Flash Attention with online softmax. Process Q in blocks; for each Q-block iterate over K/V blocks. Track running max and sum to rescale the accumulator without materializing the full attention matrix.

### Formula
For each new block with max $m_{\text{new}}$:
$$\text{acc} \leftarrow \text{acc} \cdot e^{m_{\text{old}} - m_{\text{new}}} + e^{s - m_{\text{new}}} V, \quad \text{out} = \text{acc} / \text{row\_sum}$$

In [ ]:
import torch
import math

def flash_attention(Q, K, V, block_size=32):
    B, S, D = Q.shape
    output = torch.zeros_like(Q)
    for i in range(0, S, block_size):
        qi = Q[:, i:i+block_size]
        bs_q = qi.shape[1]
        row_max = torch.full((B, bs_q, 1), float('-inf'), device=Q.device)
        row_sum = torch.zeros(B, bs_q, 1, device=Q.device)
        acc = torch.zeros(B, bs_q, D, device=Q.device)
        for j in range(0, S, block_size):
            kj = K[:, j:j+block_size]
            vj = V[:, j:j+block_size]
            scores = torch.bmm(qi, kj.transpose(1, 2)) / math.sqrt(D)
            block_max = scores.max(dim=-1, keepdim=True).values
            new_max = torch.maximum(row_max, block_max)
            correction = torch.exp(row_max - new_max)
            exp_scores = torch.exp(scores - new_max)
            acc = acc * correction + torch.bmm(exp_scores, vj)
            row_sum = row_sum * correction + exp_scores.sum(dim=-1, keepdim=True)
            row_max = new_max
        output[:, i:i+block_size] = acc / row_sum
    return output

In [ ]:
import torch
import math

# --- Test 1: Matches standard attention ---
torch.manual_seed(0)
B, S, D = 2, 16, 8
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(D)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 2: Non-aligned block size ---
torch.manual_seed(42)
B, S, D = 1, 7, 4
Q, K, V = torch.randn(B,S,D), torch.randn(B,S,D), torch.randn(B,S,D)
out = flash_attention(Q, K, V, block_size=3)
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(D)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 3: Block size invariant ---
torch.manual_seed(0)
Q, K, V = torch.randn(1,12,8), torch.randn(1,12,8), torch.randn(1,12,8)
assert torch.allclose(flash_attention(Q, K, V, block_size=4),
                      flash_attention(Q, K, V, block_size=6), atol=1e-4)

# --- Test 4: Gradient flow ---
Q = torch.randn(1, 8, 4, requires_grad=True)
K = torch.randn(1, 8, 4, requires_grad=True)
V = torch.randn(1, 8, 4, requires_grad=True)
flash_attention(Q, K, V, block_size=4).sum().backward()
assert Q.grad is not None

print('All tests passed!')

## Non-causal Flash-style KV Cache GQA

In [4]:
import torch
import torch.nn as nn
import math


class KVCacheNonCausalGroupQueryFlashAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int,
                 max_seq_len: int, block_size: int = 256):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.repeats = num_heads // num_kv_heads
        self.max_seq_len = max_seq_len
        self.block_size = block_size

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

        self.cache_k = None   # (B, H_kv, max_seq_len, d_k)
        self.cache_v = None   # (B, H_kv, max_seq_len, d_k)
        self.cache_len = 0

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.cache_len = 0

    def repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, H_kv, S, d_k)
        return: (B, H, S, d_k)
        """
        B, H_kv, S, D = x.shape
        x = x.unsqueeze(2)                                   # (B, H_kv, 1, S, D)
        x = x.expand(B, H_kv, self.repeats, S, D)           # (B, H_kv, repeats, S, D)
        x = x.contiguous().view(B, H_kv * self.repeats, S, D)
        return x

    def flash_attention_blockwise_noncausal(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor
    ) -> torch.Tensor:
        """
        Educational FlashAttention-style non-causal attention.

        q: (B, H, S_q, D)
        k: (B, H, S_k, D)
        v: (B, H, S_k, D)

        return:
            out: (B, H, S_q, D)
        """
        B, H, S_q, D = q.shape
        S_k = k.shape[2]
        scale = math.sqrt(D)

        # running max, denominator, numerator accumulator
        m = torch.full((B, H, S_q), float("-inf"), device=q.device, dtype=q.dtype)
        l = torch.zeros((B, H, S_q), device=q.device, dtype=q.dtype)
        out = torch.zeros((B, H, S_q, D), device=q.device, dtype=q.dtype)

        for start in range(0, S_k, self.block_size):
            end = min(start + self.block_size, S_k)

            k_block = k[:, :, start:end, :]   # (B, H, B_k, D)
            v_block = v[:, :, start:end, :]   # (B, H, B_k, D)

            # local scores: (B, H, S_q, B_k)
            scores = torch.matmul(q, k_block.transpose(-2, -1)) / scale

            # online softmax update
            block_row_max = scores.max(dim=-1).values                       # (B, H, S_q)
            m_new = torch.maximum(m, block_row_max)                         # (B, H, S_q)

            correction = torch.exp(m - m_new)                               # (B, H, S_q)
            P = torch.exp(scores - m_new.unsqueeze(-1))                     # (B, H, S_q, B_k)

            l_new = correction * l + P.sum(dim=-1)                          # (B, H, S_q)
            out = correction.unsqueeze(-1) * out + torch.matmul(P, v_block) # (B, H, S_q, D)

            m = m_new
            l = l_new

        out = out / l.unsqueeze(-1)
        return out

    def forward_prefill(self, x: torch.Tensor) -> torch.Tensor:
        """
        Non-causal prefill over the whole prompt and fill KV cache.

        x: (B, S, d_model)
        return: (B, S, d_model)
        """
        B, S, _ = x.shape
        assert S <= self.max_seq_len

        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)       # (B, H,   S, d_k)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)    # (B, Hkv, S, d_k)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)    # (B, Hkv, S, d_k)

        # fill cache
        self.cache_k = torch.zeros(
            B, self.num_kv_heads, self.max_seq_len, self.d_k,
            device=x.device, dtype=x.dtype
        )
        self.cache_v = torch.zeros(
            B, self.num_kv_heads, self.max_seq_len, self.d_k,
            device=x.device, dtype=x.dtype
        )
        self.cache_k[:, :, :S, :] = k
        self.cache_v[:, :, :S, :] = v
        self.cache_len = S

        k_full = self.repeat_kv(k)   # (B, H, S, d_k)
        v_full = self.repeat_kv(v)   # (B, H, S, d_k)

        out = self.flash_attention_blockwise_noncausal(q, k_full, v_full)
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        out = self.W_o(out)
        return out

    def forward_decode(self, x_t: torch.Tensor) -> torch.Tensor:
        """
        Decode one step using KV cache.

        x_t: (B, 1, d_model)
        return: (B, 1, d_model)
        """
        B, S_new, _ = x_t.shape
        assert S_new == 1, "forward_decode expects one token at a time"

        q = self.W_q(x_t).view(B, 1, self.num_heads, self.d_k).transpose(1, 2)        # (B, H,   1, d_k)
        k_new = self.W_k(x_t).view(B, 1, self.num_kv_heads, self.d_k).transpose(1, 2) # (B, Hkv, 1, d_k)
        v_new = self.W_v(x_t).view(B, 1, self.num_kv_heads, self.d_k).transpose(1, 2) # (B, Hkv, 1, d_k)

        if self.cache_k is None or self.cache_v is None:
            self.cache_k = torch.zeros(
                B, self.num_kv_heads, self.max_seq_len, self.d_k,
                device=x_t.device, dtype=x_t.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_kv_heads, self.max_seq_len, self.d_k,
                device=x_t.device, dtype=x_t.dtype
            )
            self.cache_len = 0

        assert self.cache_len < self.max_seq_len

        pos = self.cache_len
        self.cache_k[:, :, pos:pos+1, :] = k_new
        self.cache_v[:, :, pos:pos+1, :] = v_new
        self.cache_len += 1

        k_cached = self.cache_k[:, :, :self.cache_len, :]   # (B, Hkv, T, d_k)
        v_cached = self.cache_v[:, :, :self.cache_len, :]   # (B, Hkv, T, d_k)

        k_full = self.repeat_kv(k_cached)   # (B, H, T, d_k)
        v_full = self.repeat_kv(v_cached)   # (B, H, T, d_k)

        out = self.flash_attention_blockwise_noncausal(q, k_full, v_full)  # (B, H, 1, d_k)
        out = out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        out = self.W_o(out)
        return out

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        """
        use_cache=False -> prefill
        use_cache=True  -> decode one token
        """
        if use_cache:
            return self.forward_decode(x)
        return self.forward_prefill(x)

torch.manual_seed(0)

model = KVCacheNonCausalGroupQueryFlashAttention(
    d_model=32,
    num_heads=4,
    num_kv_heads=2,
    max_seq_len=16,
    block_size=4,
)

x = torch.randn(2, 6, 32)
out = model.forward_prefill(x)
print(out.shape)      # torch.Size([2, 6, 32])
print(model.cache_len)  # 6

x_t = torch.randn(2, 1, 32)
out_t = model.forward_decode(x_t)
print(out_t.shape)    # torch.Size([2, 1, 32])
print(model.cache_len)  # 7


torch.Size([2, 6, 32])
6
torch.Size([2, 1, 32])
7


### Causal Flash-attn

In [3]:
import torch
import torch.nn as nn
import math


class KVCacheCausalGroupQueryFlashAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int,
                 max_seq_len: int, block_size: int = 256):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.repeats = num_heads // num_kv_heads
        self.max_seq_len = max_seq_len
        self.block_size = block_size

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

        # cache
        self.cache_k = None   # (B, H_kv, max_seq_len, d_k)
        self.cache_v = None   # (B, H_kv, max_seq_len, d_k)
        self.cache_len = 0

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.cache_len = 0

    def repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, H_kv, S, d_k)
        return: (B, H, S, d_k)
        """
        B, H_kv, S, D = x.shape
        x = x.unsqueeze(2)                                   # (B, H_kv, 1, S, D)
        x = x.expand(B, H_kv, self.repeats, S, D)           # (B, H_kv, repeats, S, D)
        x = x.contiguous().view(B, H_kv * self.repeats, S, D)
        return x

    def flash_attention_blockwise_causal(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        q_start_pos: int = 0
    ) -> torch.Tensor:
        """
        Educational FlashAttention-style causal attention with online softmax.

        q: (B, H, S_q, D)
        k: (B, H, S_k, D)
        v: (B, H, S_k, D)
        q_start_pos:
            absolute position of q[:, :, 0, :] in the sequence.
            Example:
              - prefill full prompt: q_start_pos = 0
              - decode one token at cache_len-1: q_start_pos = cache_len-1

        return:
            out: (B, H, S_q, D)
        """
        B, H, S_q, D = q.shape
        S_k = k.shape[2]
        scale = math.sqrt(D)

        # Running values per query row
        # m: running max
        # l: running denominator
        # out: running numerator accumulator
        m = torch.full((B, H, S_q), float("-inf"), device=q.device, dtype=q.dtype)
        l = torch.zeros((B, H, S_q), device=q.device, dtype=q.dtype)
        out = torch.zeros((B, H, S_q, D), device=q.device, dtype=q.dtype)

        # absolute query positions
        q_pos = torch.arange(q_start_pos, q_start_pos + S_q, device=q.device)   # (S_q,)

        # loop over key/value blocks
        for start in range(0, S_k, self.block_size):
            end = min(start + self.block_size, S_k)

            k_block = k[:, :, start:end, :]   # (B, H, B_k, D)
            v_block = v[:, :, start:end, :]   # (B, H, B_k, D)

            # local scores: (B, H, S_q, B_k)
            scores = torch.matmul(q, k_block.transpose(-2, -1)) / scale

            # causal mask for this block
            k_pos = torch.arange(start, end, device=q.device)                   # (B_k,)
            causal_mask = k_pos.unsqueeze(0) <= q_pos.unsqueeze(1)              # (S_q, B_k)

            # broadcast to (B, H, S_q, B_k)
            scores = scores.masked_fill(~causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))

            # block row max
            block_row_max = scores.max(dim=-1).values                           # (B, H, S_q)

            # new running max
            m_new = torch.maximum(m, block_row_max)                             # (B, H, S_q)

            # correction factor for old accumulator/denominator
            correction = torch.exp(m - m_new)                                   # (B, H, S_q)

            # exponentials under the new max
            P = torch.exp(scores - m_new.unsqueeze(-1))                         # (B, H, S_q, B_k)

            # update denominator
            l_new = correction * l + P.sum(dim=-1)                              # (B, H, S_q)

            # update numerator accumulator
            out = correction.unsqueeze(-1) * out + torch.matmul(P, v_block)     # (B, H, S_q, D)

            # save running state
            m = m_new
            l = l_new

        # final normalization
        out = out / l.unsqueeze(-1)
        return out

    def forward_prefill(self, x: torch.Tensor) -> torch.Tensor:
        """
        Prefill stage: process the whole prompt at once and fill KV cache.

        x: (B, S, d_model)
        return: (B, S, d_model)
        """
        B, S, _ = x.shape
        assert S <= self.max_seq_len

        # projections
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)       # (B, H,   S, d_k)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)    # (B, Hkv, S, d_k)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)    # (B, Hkv, S, d_k)

        # save to cache
        self.cache_k = torch.zeros(
            B, self.num_kv_heads, self.max_seq_len, self.d_k,
            device=x.device, dtype=x.dtype
        )
        self.cache_v = torch.zeros(
            B, self.num_kv_heads, self.max_seq_len, self.d_k,
            device=x.device, dtype=x.dtype
        )
        self.cache_k[:, :, :S, :] = k
        self.cache_v[:, :, :S, :] = v
        self.cache_len = S

        # expand kv heads
        k_full = self.repeat_kv(k)   # (B, H, S, d_k)
        v_full = self.repeat_kv(v)   # (B, H, S, d_k)

        # flash-style causal attention
        out = self.flash_attention_blockwise_causal(
            q=q,
            k=k_full,
            v=v_full,
            q_start_pos=0
        )  # (B, H, S, d_k)

        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        out = self.W_o(out)
        return out

    def forward_decode(self, x_t: torch.Tensor) -> torch.Tensor:
        """
        Decode one step using KV cache.

        x_t: (B, 1, d_model)
        return: (B, 1, d_model)
        """
        B, S_new, _ = x_t.shape
        assert S_new == 1, "forward_decode expects one token at a time"

        # current q, k, v
        q = self.W_q(x_t).view(B, 1, self.num_heads, self.d_k).transpose(1, 2)        # (B, H,   1, d_k)
        k_new = self.W_k(x_t).view(B, 1, self.num_kv_heads, self.d_k).transpose(1, 2) # (B, Hkv, 1, d_k)
        v_new = self.W_v(x_t).view(B, 1, self.num_kv_heads, self.d_k).transpose(1, 2) # (B, Hkv, 1, d_k)

        # initialize cache if empty
        if self.cache_k is None or self.cache_v is None:
            self.cache_k = torch.zeros(
                B, self.num_kv_heads, self.max_seq_len, self.d_k,
                device=x_t.device, dtype=x_t.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_kv_heads, self.max_seq_len, self.d_k,
                device=x_t.device, dtype=x_t.dtype
            )
            self.cache_len = 0

        assert self.cache_len < self.max_seq_len

        # append new kv into cache
        pos = self.cache_len
        self.cache_k[:, :, pos:pos+1, :] = k_new
        self.cache_v[:, :, pos:pos+1, :] = v_new
        self.cache_len += 1

        # read valid cached kv only
        k_cached = self.cache_k[:, :, :self.cache_len, :]   # (B, Hkv, T, d_k)
        v_cached = self.cache_v[:, :, :self.cache_len, :]   # (B, Hkv, T, d_k)

        # expand kv heads
        k_full = self.repeat_kv(k_cached)   # (B, H, T, d_k)
        v_full = self.repeat_kv(v_cached)   # (B, H, T, d_k)

        # flash-style causal attention
        # current token has absolute position cache_len - 1
        out = self.flash_attention_blockwise_causal(
            q=q,
            k=k_full,
            v=v_full,
            q_start_pos=self.cache_len - 1
        )  # (B, H, 1, d_k)

        out = out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        out = self.W_o(out)
        return out

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        """
        Convenience wrapper:
        - use_cache=False -> prefill full sequence
        - use_cache=True  -> decode one token
        """
        if use_cache:
            return self.forward_decode(x)
        return self.forward_prefill(x)

torch.manual_seed(0)

model = KVCacheCausalGroupQueryFlashAttention(
    d_model=32,
    num_heads=4,
    num_kv_heads=2,
    max_seq_len=16,
    block_size=4
)

# prefill
x = torch.randn(2, 6, 32)
out = model.forward_prefill(x)
print(out.shape)      # torch.Size([2, 6, 32])
print(model.cache_len)  # 6

# decode one token
x_t = torch.randn(2, 1, 32)
out_t = model.forward_decode(x_t)
print(out_t.shape)    # torch.Size([2, 1, 32])
print(model.cache_len)  # 7


torch.Size([2, 6, 32])
6
torch.Size([2, 1, 32])
7
